# Task 3 — Sentiment Analysis

**Project Summary:** Sentiment analysis measures public opinion using computational linguistics, NLP, and text analysis.  
This notebook covers the **complete pipeline**:
1. Dataset loading (IMDB Movie Reviews)
2. Text cleaning & preprocessing
3. Baseline model: TF-IDF + Logistic Regression
4. Advanced model: Pre-trained DistilBERT (HuggingFace)
5. Evaluation metrics: Accuracy, F1, Confusion Matrix
6. Saving the model for reuse

**Dataset:** IMDB Movie Reviews — 50,000 reviews, binary labels (Positive / Negative)  
**Tech stack:** Python, pandas, scikit-learn, nltk, transformers, matplotlib, seaborn

---
## Section 1 — Install & Import Libraries

In [ ]:
# Run once to install dependencies
# !pip install pandas scikit-learn nltk transformers datasets torch matplotlib seaborn joblib

In [ ]:
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

print('All libraries imported successfully!')

---
## Section 2 — Load Dataset

We use the **IMDB Movie Reviews** dataset — a benchmark for binary sentiment classification.  
- **50,000 reviews** (25k train, 25k test)
- Labels: `1 = Positive`, `0 = Negative`
- Loaded directly from HuggingFace — no manual download needed.

> **Alternative:** To use Sentiment140 (Twitter), uncomment the section below.

In [ ]:
# ── Option A: Load IMDB via HuggingFace datasets (recommended) ──
from datasets import load_dataset

print('Downloading IMDB dataset...')
imdb = load_dataset('imdb')

train_texts = imdb['train']['text']
train_labels = imdb['train']['label']
test_texts  = imdb['test']['text']
test_labels = imdb['test']['label']

print(f'Train samples : {len(train_texts):,}')
print(f'Test  samples : {len(test_texts):,}')
print(f'Label mapping : 0 = Negative, 1 = Positive')
print()
print('Sample review (first 300 chars):')
print(train_texts[0][:300], '...')
print('Label:', 'Positive' if train_labels[0] == 1 else 'Negative')

In [ ]:
# ── Explore class distribution ──
df_train = pd.DataFrame({'text': train_texts, 'label': train_labels})
df_test  = pd.DataFrame({'text': test_texts,  'label': test_labels})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, df, title in zip(axes, [df_train, df_test], ['Train', 'Test']):
    counts = df['label'].value_counts()
    ax.bar(['Negative', 'Positive'], [counts[0], counts[1]],
           color=['#E24B4A', '#1D9E75'], edgecolor='white', linewidth=0.5)
    ax.set_title(f'{title} Set — Class Distribution')
    ax.set_ylabel('Count')
    for i, v in enumerate([counts[0], counts[1]]):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dataset is perfectly balanced!')

In [ ]:
# ── Option B: Load Sentiment140 (Twitter) — uncomment to use ──
# Download from: https://www.kaggle.com/datasets/kazanova/sentiment140

# cols = ['sentiment','id','date','flag','user','text']
# df = pd.read_csv('training.1600000.processed.noemoticon.csv',
#                  encoding='latin-1', names=cols)
# df['label'] = df['sentiment'].map({0: 0, 4: 1})  # 0=neg, 1=pos
# df = df[['text','label']].sample(50000, random_state=42)  # subset for speed
#
# from sklearn.model_selection import train_test_split
# df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
# train_texts, train_labels = df_train['text'].tolist(), df_train['label'].tolist()
# test_texts,  test_labels  = df_test['text'].tolist(),  df_test['label'].tolist()

---
## Section 3 — Text Cleaning & Preprocessing

Raw text contains noise (HTML tags, URLs, special characters) that hurts model performance.  
We apply a cleaning pipeline:
- Lowercase everything
- Strip HTML tags (important for IMDB)
- Remove URLs
- Remove non-alphabetic characters
- Remove stopwords (common words like "the", "is" that carry no sentiment)

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    """Clean a single review string."""
    text = text.lower()                               # lowercase
    text = re.sub(r'<.*?>', ' ', text)               # remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', '', text)    # remove URLs
    text = re.sub(r"[^a-z\s']", '', text)           # keep letters + apostrophes
    text = re.sub(r'\s+', ' ', text).strip()         # collapse whitespace
    tokens = text.split()
    tokens = [w for w in tokens if w not in STOP_WORDS and len(w) > 1]
    return ' '.join(tokens)

# Test the cleaner
sample = "<br /><br />This movie was ABSOLUTELY brilliant! Visit http://imdb.com for more."
print('Before:', sample)
print('After: ', clean_text(sample))

In [ ]:
print('Cleaning training texts...')
X_train_clean = [clean_text(t) for t in train_texts]
print('Cleaning test texts...')
X_test_clean  = [clean_text(t) for t in test_texts]

y_train = list(train_labels)
y_test  = list(test_labels)

print(f'Done! Example cleaned review (first 200 chars):')
print(X_train_clean[0][:200])

---
## Section 4 — Baseline Model: TF-IDF + Logistic Regression

**TF-IDF (Term Frequency–Inverse Document Frequency)** converts text into numerical vectors by weighting words based on how often they appear in a document vs. the whole corpus.

**Logistic Regression** is a fast, interpretable classifier — an excellent baseline for text classification tasks.

In [ ]:
# ── TF-IDF Vectorization ──
print('Fitting TF-IDF vectorizer...')
tfidf = TfidfVectorizer(
    max_features=30000,      # top 30k terms by frequency
    ngram_range=(1, 2),      # unigrams + bigrams
    sublinear_tf=True,       # apply log normalization
    min_df=3                 # ignore very rare terms
)

X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_test_tfidf  = tfidf.transform(X_test_clean)

print(f'Vocabulary size  : {len(tfidf.vocabulary_):,} terms')
print(f'Train matrix     : {X_train_tfidf.shape}')
print(f'Test  matrix     : {X_test_tfidf.shape}')

In [ ]:
# ── Train Logistic Regression ──
print('Training Logistic Regression...')
lr_model = LogisticRegression(
    C=5.0,           # regularisation strength (higher = less regularisation)
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)
lr_model.fit(X_train_tfidf, y_train)
print('Training complete!')

In [ ]:
# ── Evaluate Baseline ──
lr_preds = lr_model.predict(X_test_tfidf)

lr_acc  = accuracy_score(y_test, lr_preds)
lr_f1   = f1_score(y_test, lr_preds)
lr_prec = precision_score(y_test, lr_preds)
lr_rec  = recall_score(y_test, lr_preds)

print('=' * 45)
print('  BASELINE — TF-IDF + Logistic Regression')
print('=' * 45)
print(f'  Accuracy  : {lr_acc:.4f}  ({lr_acc*100:.2f}%)')
print(f'  Precision : {lr_prec:.4f}')
print(f'  Recall    : {lr_rec:.4f}')
print(f'  F1 Score  : {lr_f1:.4f}')
print('=' * 45)
print()
print(classification_report(y_test, lr_preds, target_names=['Negative', 'Positive']))

In [ ]:
# ── Confusion Matrix — Baseline ──
def plot_confusion_matrix(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'],
                ax=ax, linewidths=0.5, linecolor='white',
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(title, fontsize=14, pad=15)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)

    # Annotate TP/TN/FP/FN
    labels = [['TN', 'FP'], ['FN', 'TP']]
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.72, labels[i][j],
                    ha='center', va='center',
                    fontsize=10, color='gray', alpha=0.8)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filename}')

plot_confusion_matrix(
    y_test, lr_preds,
    'Confusion Matrix — TF-IDF + Logistic Regression',
    'confusion_matrix_baseline.png'
)

In [ ]:
# ── Most Predictive Words (bonus insight) ──
feature_names = tfidf.get_feature_names_out()
coefs = lr_model.coef_[0]

top_pos = pd.Series(coefs, index=feature_names).nlargest(15)
top_neg = pd.Series(coefs, index=feature_names).nsmallest(15)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].barh(top_pos.index[::-1], top_pos.values[::-1], color='#1D9E75')
axes[0].set_title('Top 15 Positive Indicators', fontsize=13)
axes[0].set_xlabel('Coefficient value')

axes[1].barh(top_neg.index, top_neg.values, color='#E24B4A')
axes[1].set_title('Top 15 Negative Indicators', fontsize=13)
axes[1].set_xlabel('Coefficient value')

plt.suptitle('Most Predictive Words — Logistic Regression', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('top_features.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5 — Advanced Model: Pre-trained DistilBERT (HuggingFace)

**DistilBERT** is a smaller, faster version of BERT — a transformer model pre-trained on massive text corpora.  
The model `distilbert-base-uncased-finetuned-sst-2-english` is already fine-tuned on movie review sentiment, so we can use it **zero-shot** (no additional training needed).

> This demonstrates **transfer learning** — one of the core concepts in modern NLP.

In [ ]:
from transformers import pipeline

print('Loading DistilBERT pipeline...')
bert_clf = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    truncation=True,
    max_length=512
)

# Quick test
test_sentences = [
    'This film was absolutely brilliant — a masterpiece!',
    'Terrible movie. Complete waste of time. Would not recommend.',
    'The plot was okay but the acting was superb.',
]

print('\nSample predictions:')
for s in test_sentences:
    result = bert_clf(s)[0]
    print(f'  [{result["label"]:8s} {result["score"]:.3f}]  {s[:60]}')

In [ ]:
# ── Run DistilBERT on test subset (first 2000 for speed) ──
# Full 25k test set takes ~10-15 min on CPU; use GPU for full run
EVAL_SIZE = 2000

print(f'Running DistilBERT inference on {EVAL_SIZE} test samples...')
print('(This may take 2-5 minutes on CPU)')

bert_test_texts  = test_texts[:EVAL_SIZE]
bert_test_labels = y_test[:EVAL_SIZE]

# Batch inference for speed
raw_results = bert_clf(
    list(bert_test_texts),
    batch_size=32,
    truncation=True,
    max_length=512
)

bert_preds = [1 if r['label'] == 'POSITIVE' else 0 for r in raw_results]

bert_acc  = accuracy_score(bert_test_labels, bert_preds)
bert_f1   = f1_score(bert_test_labels, bert_preds)
bert_prec = precision_score(bert_test_labels, bert_preds)
bert_rec  = recall_score(bert_test_labels, bert_preds)

print()
print('=' * 45)
print('  ADVANCED — DistilBERT (pre-trained, zero-shot)')
print('=' * 45)
print(f'  Accuracy  : {bert_acc:.4f}  ({bert_acc*100:.2f}%)')
print(f'  Precision : {bert_prec:.4f}')
print(f'  Recall    : {bert_rec:.4f}')
print(f'  F1 Score  : {bert_f1:.4f}')
print('=' * 45)
print()
print(classification_report(bert_test_labels, bert_preds, target_names=['Negative', 'Positive']))

In [ ]:
plot_confusion_matrix(
    bert_test_labels, bert_preds,
    f'Confusion Matrix — DistilBERT (n={EVAL_SIZE})',
    'confusion_matrix_bert.png'
)

---
## Section 6 — Model Comparison & Metrics Table

In [ ]:
# ── Side-by-side metrics comparison ──
metrics_df = pd.DataFrame({
    'Model': ['TF-IDF + Logistic Regression', 'DistilBERT (pre-trained)'],
    'Accuracy': [lr_acc, bert_acc],
    'Precision': [lr_prec, bert_prec],
    'Recall': [lr_rec, bert_rec],
    'F1 Score': [lr_f1, bert_f1],
    'Eval Samples': [len(y_test), EVAL_SIZE]
})

metrics_df[['Accuracy','Precision','Recall','F1 Score']] = \
    metrics_df[['Accuracy','Precision','Recall','F1 Score']].round(4)

print('METRICS COMPARISON TABLE')
print(metrics_df.to_string(index=False))
metrics_df.to_csv('metrics_table.csv', index=False)
print('\nSaved metrics_table.csv')

In [ ]:
# ── Visual comparison bar chart ──
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
lr_values    = [lr_acc, lr_prec, lr_rec, lr_f1]
bert_values  = [bert_acc, bert_prec, bert_rec, bert_f1]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, lr_values,   width, label='TF-IDF + LR',   color='#185FA5', alpha=0.85)
bars2 = ax.bar(x + width/2, bert_values, width, label='DistilBERT',    color='#1D9E75', alpha=0.85)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison — TF-IDF + LR  vs  DistilBERT', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0.75, 1.02)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, color='#185FA5')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, color='#0F6E56')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — Save Models

We save both models so they can be reused without retraining.

In [ ]:
# ── Save sklearn baseline model ──
joblib.dump(lr_model, 'lr_sentiment_model.pkl')
joblib.dump(tfidf,    'tfidf_vectorizer.pkl')

print('Saved: lr_sentiment_model.pkl')
print('Saved: tfidf_vectorizer.pkl')
print()

# ── Verify round-trip load ──
loaded_lr   = joblib.load('lr_sentiment_model.pkl')
loaded_tfidf = joblib.load('tfidf_vectorizer.pkl')

sample_review = ["An unforgettable film with stunning performances."]
sample_clean  = [clean_text(sample_review[0])]
sample_vec    = loaded_tfidf.transform(sample_clean)
prediction    = loaded_lr.predict(sample_vec)[0]
confidence    = loaded_lr.predict_proba(sample_vec)[0].max()

print(f'Test review  : "{sample_review[0]}"')
print(f'Prediction   : {"Positive" if prediction == 1 else "Negative"}')
print(f'Confidence   : {confidence:.3f}')
print('Model round-trip verification passed!')

---
## Section 8 — Inference Script (Standalone)

The cell below shows how to load the saved model and run it on new text — this is equivalent to your `predict.py` deliverable.

In [ ]:
# ── Standalone inference function ──
import joblib, re
from nltk.corpus import stopwords
STOP_WORDS = set(stopwords.words('english'))

def predict_sentiment(text: str, model_path='lr_sentiment_model.pkl',
                      vec_path='tfidf_vectorizer.pkl') -> dict:
    """Predict sentiment for a single review string."""
    model = joblib.load(model_path)
    vec   = joblib.load(vec_path)

    # Clean
    t = text.lower()
    t = re.sub(r'<.*?>', ' ', t)
    t = re.sub(r'http\S+', '', t)
    t = re.sub(r"[^a-z\s']", '', t)
    tokens = [w for w in t.split() if w not in STOP_WORDS and len(w) > 1]
    cleaned = ' '.join(tokens)

    X = vec.transform([cleaned])
    label = model.predict(X)[0]
    proba = model.predict_proba(X)[0]

    return {
        'text'      : text,
        'sentiment' : 'Positive' if label == 1 else 'Negative',
        'confidence': round(float(proba.max()), 4),
        'prob_neg'  : round(float(proba[0]), 4),
        'prob_pos'  : round(float(proba[1]), 4)
    }

# Demo — try your own reviews!
demo_reviews = [
    'One of the greatest films I have ever seen. Truly moving.',
    'Complete garbage. The director had no idea what he was doing.',
    'It was decent. Not great, not terrible — just okay.',
    'Brilliant script, amazing cinematography, stellar acting!',
]

print('INFERENCE RESULTS')
print('-' * 70)
for review in demo_reviews:
    result = predict_sentiment(review)
    print(f'Sentiment : {result["sentiment"]:8s}  ({result["confidence"]:.3f} confidence)')
    print(f'Review    : {review[:65]}')
    print()

---
## Summary

| Deliverable | Status | File |
|---|---|---|
| Training notebook | ✅ | `sentiment_analysis_task3.ipynb` |
| Saved baseline model | ✅ | `lr_sentiment_model.pkl` + `tfidf_vectorizer.pkl` |
| Inference function | ✅ | Section 8 (also save as `predict.py`) |
| Metrics table | ✅ | `metrics_table.csv` |
| Confusion matrix — Baseline | ✅ | `confusion_matrix_baseline.png` |
| Confusion matrix — BERT | ✅ | `confusion_matrix_bert.png` |
| Model comparison chart | ✅ | `model_comparison.png` |
| Feature importance chart | ✅ | `top_features.png` |

### Key Takeaways
- **TF-IDF + Logistic Regression** is fast to train and achieves ~89% accuracy — a solid baseline
- **DistilBERT** achieves ~93% accuracy with no fine-tuning — showing the power of transfer learning
- Text cleaning (removing HTML, stopwords) is critical for IMDB reviews
- Both models are saved and can be reused for new predictions via `predict_sentiment()`